← [13b - UNSAT cores : le MUS](Z3-13b-UnsatCores-MUS-CSharp.ipynb) | [README Z3-API](README.md) | [15 - Tableaux imbriqués (Python) >>](Z3-15-Nested-Arrays-2D-Python.ipynb) | [15 - Tableaux imbriqués (C#)](Z3-15-Nested-Arrays-2D-CSharp.ipynb)

# 14. Bit-vectors : verifier le debordement arithmetique
Jusqu'ici nous avons raisonne sur des **entiers non bornes** (`Int`) ou des **reels exacts** (`Real`). Mais un ordinateur calcule sur des **entiers bornes** : un `uint32` ne peut pas depasser `2^32 - 1`, et l'addition **enveloppe** (wraps) modulo `2^32` quand la somme depasse. Ce debordement (overflow) est la source de bugs legendaires (Ariane 5, OpenSSL, Pac-Man) et la cible centrale de la **verification formelle** de code.
Z3 traite les entiers machines via la théorie des **bit-vectors** (`BitVec`, SMT-LIB `BV`) : un vecteur de *n* bits interprete comme un entier modulo 2^n, avec addition/multiplication/comparaisons/bit-a-bit **tous decidables**. Ce notebook demontre la capacite distinctive de cette théorie : **prouver** qu'un debordement est inevitable (ou impossible) sur le domaine machine tout entier - pas seulement le constater sur des valeurs fixes.
> Port C# du notebook Python [`Z3-14-BitVectors-Overflow-Python.ipynb`](Z3-14-BitVectors-Overflow-Python.ipynb). (Le bloc DSL `[BitVecWidth(n)]` de la serie Z3.Linq, specifique a cette couche declarative, n'est pas porte ; l'API imperative brute `Microsoft.Z3` fait le même travail.)
## Source

Ce notebook s'inscrit dans la série ancrée par **Yurichev, Dennis — *SAT/SMT by Example* (édition publique du 12 février 2023, 666 pages)**, §5.1 Integer overflow (p. 125), Ch. 5 Verification — Bit-vectors, overflow, ULT/UGE. Gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16,45 Mo, magic bytes `%PDF`). Voir [Z3-API/README.md — Ouvrage de référence](README.md#ouvrage-de-référence) pour l'index chapitré complet de la série.

In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);
Console.WriteLine("Imports OK : Microsoft.Z3 version " + Microsoft.Z3.Version.FullVersion);


Installing Packages Microsoft.Z3

Imports OK : Microsoft.Z3 version Z3 4.12.2.0


### Lecture : une théorie de décision, pas une approximation

`using Microsoft.Z3;` ne fixe aucune théorie : c'est la **sorte** des variables qui la
choisit. Ce notebook travaille en `BitVec(..., 32)`, c'est-à-dire en arithmétique
modulaire sur 32 bits, où l'addition est exacte *modulo* `2^32`. C'est ce qui le
distingue du notebook 12 : là-bas `Real` donnait des rationnels exacts et non bornés,
ici les valeurs sont bornées par construction et l'enveloppement n'est pas une erreur
d'arrondi mais la sémantique même de l'opération.

La conséquence est que Z3 reste une **théorie de décision** dans ce fragment : tout
énoncé du langage des bit-vectors admet une réponse `sat` ou `unsat` en temps fini, et
cette réponse ne dépend ni d'un pas d'itération ni d'un échantillonnage. C'est ce qui
autorise à *prouver* des propriétés de débordement plutôt qu'à les tester.

## 1. La théorie des bit-vectors (SMT-LIB `BitVec`)
Un **bit-vector** est une suite de *n* bits interpretee comme un entier modulo 2^n. Z3 en fait une **théorie de decision** (pas une approximation) : l'addition, la multiplication, les comparaisons et les opérations bit-a-bit sont définies modulo 2^n et decidables. En C#, `MkBVConst("x", 32)` crée un vecteur de 32 bits, `MkBV(v, 32)` une valeur, `MkBVULT`/`MkBVUGE` les comparaisons **non signees**, `MkExtract(hi, lo, x)` l'extraction de champ.
La distinction cle : `Int` (non borne, `a + b >= a` toujours vrai pour `b >= 0`) vs `BitVec` (borne, `a + b` peut envelopper et devenir `< a`). C'est cette borne fixe qui rend le **debordement expressible**- et decidable.

## 2. Le debordement se produit (et Z3 le calcule)

Additionnons deux valeurs dont la somme mathematique **depasse 2^32**. En `uint32`, `3 000 000 000 + 1 500 000 000 = 4 500 000 000` mathematiquement, mais la valeur machine **enveloppe** (retranche 2^32). Z3 calcule la valeur exacte après enveloppement.

In [2]:
// Deux valeurs dont la somme depasse 2^32 = 4 294 967 296
var ctx = new Context();
var a = ctx.MkBV(3000000000u, 32);   // 3 milliards : tient en uint32
var b = ctx.MkBV(1500000000u, 32);   // 1,5 milliard
var somme = (BitVecExpr)ctx.MkBVAdd(a, b).Simplify();  // addition modulo 2^32
ulong aValeur = ((BitVecNum)a).UInt64;
ulong bValeur = ((BitVecNum)b).UInt64;
ulong sommeValeur = ((BitVecNum)somme).UInt64;
Console.WriteLine($"a = {aValeur}");
Console.WriteLine($"b = {bValeur}");
Console.WriteLine($"a + b (mathematique) = {aValeur + bValeur}");
Console.WriteLine($"a + b (uint32, apres enveloppement) = {sommeValeur}");
Console.WriteLine($"-> La valeur machine ({sommeValeur}) est INFERIEURE a a ({aValeur}) : il y a eu debordement.");

a = 3000000000


b = 1500000000


a + b (mathematique) = 4500000000


a + b (uint32, apres enveloppement) = 205032704


-> La valeur machine (205032704) est INFERIEURE a a (3000000000) : il y a eu debordement.


### Lecture : le retournement du comparateur

`4 500 000 000` devient `205 032 704`, et ce n'est pas un artefact d'affichage : en
uint32 l'addition est **modulo 2^32**, donc
`4 500 000 000 - 4 294 967 296 = 205 032 704`. Le détail qui mérite qu'on s'arrête est
le dernier : la somme machine est **inférieure** à l'opérande `a`
(`205 032 704 < 3 000 000 000`). C'est la signature du débordement non signé, et c'est
ce qui rend le prédicat `somme < a` correct pour le détecter — une addition qui reste
dans les bornes ne peut pas produire une somme inférieure à l'un de ses termes positifs.

Retenez ce retournement : il fonde les deux sections suivantes, qui posent la même
inégalité — d'abord sur des valeurs fixées, puis sur des variables symboliques.

## 3. Caracteriser le debordement : `somme < a` (unsigned)

Pour des valeurs fixes, on peut demander au solveur si la condition de debordement tient. En arithmetique non signee, `somme < a` (unsigned less-than, `MkBVULT`) est vrai ssi l'addition a enveloppe : la somme "revient avant" le premier operande.

In [3]:
// Pour a, b fixes : la condition somme < a (unsigned) est-elle vraie ?
var s8 = ctx.MkSolver();
s8.Add(ctx.MkBVULT(somme, a));   // somme < a  (unsigned less-than) = debordement
var st8 = s8.Check();
Console.WriteLine($"Requete 'somme < a' pour a=3e9, b=1.5e9 : {st8}");
if (st8 == Status.SATISFIABLE)
{
    Console.WriteLine("-> SAT : MkBVULT(somme, a) tient, le debordement est confirme.");
}

Requete 'somme < a' pour a=3e9, b=1.5e9 : SATISFIABLE


-> SAT : MkBVULT(somme, a) tient, le debordement est confirme.


### Lecture : constater n'est pas prouver

`somme` est ici une **expression construite** à partir de deux valeurs concrètes, pas
une variable libre. Le `sat` répond donc à une question déjà instanciée : « pour ces
deux nombres-là, la condition de débordement tient-elle ? ». La réponse est oui, et elle
se vérifie à la main.

Cette section sert d'étalon : elle fixe la sémantique de `MkBVULT` (*unsigned less-than*,
qui compare `205 032 704` et `3 000 000 000` comme des entiers positifs) avant que la
section suivante ne passe aux variables symboliques — là où plus aucune valeur n'est
fixée et où la réponse ne se vérifie plus sur une calculette.

## 4. Prouve symbolique : `x, y >= 2^31` implique `x + y` deborde
Le vrai pouvoir de la théorie BitVec n'est pas de constater un debordement sur des valeurs fixes, mais de **prouver** qu'il est inevitable pour toute une classe d'entrees. Prouvons : pour tout `x, y >= 2^31` (la moitie superieure de l'intervalle uint32), `x + y` deborde.
Technique : par **refutation**. Si la propriete etait fausse, il existerait `x, y >= 2^31` avec `somme >= x` (pas de debordement). On demande au solveur un tel contre-exemple - s'il repond `unsat`, la propriete est **prouvee**.

In [4]:
// Pour x, y SYMBOLES (quelconques), prouvons : x,y >= 2^31  =>  x+y deborde
var x = ctx.MkBVConst("x", 32);
var y = ctx.MkBVConst("y", 32);
var sxy = ctx.MkBVAdd(x, y);

var preconditions = ctx.MkAnd(
    ctx.MkBVUGE(x, ctx.MkBV(0x80000000u, 32)),   // x >= 2^31 = 2 147 483 648
    ctx.MkBVUGE(y, ctx.MkBV(0x80000000u, 32))    // y >= 2^31
);
var pas_debord = ctx.MkBVUGE(sxy, x);             // somme >= x  = pas de debordement

// Nions la propriete : preconditions ET pas de debordement
var s2 = ctx.MkSolver();
s2.Add(preconditions);
s2.Add(pas_debord);
var st2 = s2.Check();
Console.WriteLine($"' x,y >= 2^31 ET somme >= x (pas de debord) ' : {st2}");
if (st2 == Status.UNSATISFIABLE)
{
    Console.WriteLine("  -> UNSAT : Z3 PROUVE que pour x,y >= 2^31, le debordement est inevitable.");
}
else
{
    Console.WriteLine("  -> SAT : contre-exemple trouve (le theoreme est faux).");
}

' x,y >= 2^31 ET somme >= x (pas de debord) ' : UNSATISFIABLE


  -> UNSAT : Z3 PROUVE que pour x,y >= 2^31, le debordement est inevitable.


### Lecture : le quantificateur implicite

La formule soumise au solveur est `preconditions ET pas_de_debord`, et le résultat est
`unsat`. Lisez-le comme un énoncé quantifié : il **n'existe pas** de `x, y` vérifiant à
la fois `x, y >= 2^31` et « pas de débordement ». La négation étant insatisfiable,
l'implication « `x, y >= 2^31` entraîne un débordement » est valide pour *toutes* les
valeurs.

C'est un changement de nature par rapport à la section précédente : Z3 ne teste plus un
cas, il caractérise tout un demi-espace. Le seuil `2^31 = 2 147 483 648` n'est pas
arbitraire — c'est exactement la moitié de `2^32`, donc la plus petite valeur à partir
de laquelle la somme de deux opérandes égaux sort du domaine représentable.

## 5. Prouve de surete : `p, q < 1000` implique `p + q` ne deborde pas

Symetrique du précédent : prouvons une **garantie de surete**. Pour tout `p, q < 1000`, l'addition uint32 ne deborde jamais. Même technique par refutation : s'il existait `p, q < 1000` avec debordement, le solveur les trouverait ; `unsat` prouve la surete.

In [5]:
// Propriete : si p < 1000 ET q < 1000, alors p+q ne deborde pas (somme >= p)
var p = ctx.MkBVConst("p", 32);
var q = ctx.MkBVConst("q", 32);
var spq = ctx.MkBVAdd(p, q);

var petites = ctx.MkAnd(
    ctx.MkBVULT(p, ctx.MkBV(1000u, 32)),
    ctx.MkBVULT(q, ctx.MkBV(1000u, 32))
);
var debord = ctx.MkBVULT(spq, p);   // debordement = somme < p

// Nions : petites ET debordement -> UNSAT prouve ' petites => pas de debordement '
var s3 = ctx.MkSolver();
s3.Add(petites);
s3.Add(debord);
var st3 = s3.Check();
Console.WriteLine($"' p,q < 1000 ET debordement ' : {st3}");
if (st3 == Status.UNSATISFIABLE)
{
    Console.WriteLine("  -> UNSAT : Z3 PROUVE que pour p,q < 1000, l'addition uint32 ne deborde jamais.");
}
else
{
    Console.WriteLine("  -> SAT : contre-exemple trouve (improbable).");
}

' p,q < 1000 ET debordement ' : UNSATISFIABLE


  -> UNSAT : Z3 PROUVE que pour p,q < 1000, l'addition uint32 ne deborde jamais.


### Lecture : la même preuve, retournée

Le notebook refait le raisonnement dans l'autre sens : au lieu de « au-dessus de la
borne le débordement est inévitable », il établit « en dessous de 1000 il est
impossible ». Là encore c'est un `unsat` qui porte la preuve, et là encore le seuil est
là où on l'attend : `1000 + 1000 = 2000` reste très loin de `2^32`, aucune enveloppe
n'est atteignable.

La symétrie des deux sections dit quelque chose d'utile au praticien : une même théorie
sert à **détecter** un risque et à **certifier** son absence, les deux requêtes ayant
exactement la même forme — préconditions *et* négation de la propriété. C'est ce schéma
qui se transpose à la vérification de bornes dans un parseur binaire ou un contrat
intelligent.

## 6. Combiner arithmetique et bit-a-bit : extraction de champ

La théorie BitVec ne se limite pas a l'arithmetique : elle raisonne aussi au **niveau bit**. `MkExtract(hi, lo, n)` extrait les bits `lo` a `hi` d'un vecteur. Question : existe-t-il un `n` non nul dont les deux octets de poids faible sont egaux ? Z3 produit un **temoin** concret.

In [6]:
// Combiner arithmetique et bit-a-bit : extraction de champ
var n = ctx.MkBVConst("n", 32);

// Extraire l'octet de poids faible (bits 0-7) et l'octet suivant (bits 8-15)
var octet0 = ctx.MkExtract(7, 0, n);    // bits 7..0   -> BV8
var octet1 = ctx.MkExtract(15, 8, n);   // bits 15..8  -> BV8

// Question : existe-t-il n NON NUL tel que octet0 == octet1 ?
var s4 = ctx.MkSolver();
s4.Add(ctx.MkEq(octet0, octet1));
s4.Add(ctx.MkNot(ctx.MkEq(n, ctx.MkBV(0u, 32))));   // forcer un temoin non trivial
var st4 = s4.Check();
Console.WriteLine($"' octet bas == octet suivant ' : {st4}");
if (st4 == Status.SATISFIABLE)
{
    var m4 = s4.Model;
    ulong vn = ((BitVecNum)m4.Evaluate(n)).UInt64;
    Console.WriteLine($"  Temoin : n = {vn} = 0x{vn:X8}");
    Console.WriteLine($"    octet0 (bits 7..0)  = 0x{vn & 0xFF:X2}");
    Console.WriteLine($"    octet1 (bits 15..8) = 0x{(vn >> 8) & 0xFF:X2}");
    // Verification interne : le temoin affiche doit bien verifier la contrainte
    bool temoinValide = ((vn & 0xFF) == ((vn >> 8) & 0xFF)) && (vn != 0);
    Console.WriteLine($"    Verification interne : octet0 == octet1 ET n != 0 -> {temoinValide}");
}
Console.WriteLine();
Console.WriteLine("-> Extract + egalite = raisonnement au niveau champ binaire (protocoles, formats).");

' octet bas == octet suivant ' : SATISFIABLE


  Temoin : n = 4293853441 = 0xFFEF0101


    octet0 (bits 7..0)  = 0x01


    octet1 (bits 15..8) = 0x01


    Verification interne : octet0 == octet1 ET n != 0 -> True


-> Extract + egalite = raisonnement au niveau champ binaire (protocoles, formats).


### Lecture du témoin : `0xFBFF0000`

Le solveur ne s'est pas contenté de rendre `sat` : il a fourni un **témoin** concret,
`n = 4 227 792 896 = 0xFBFF0000` dans le jumeau Python, et le carnet .NET affiche de son
côté le modèle que Z3 4.12.2 choisit — un modèle parmi d'autres, qui peut différer du
témoin Python tout en vérifiant exactement la même contrainte. Le carnet en décompose
les deux octets bas : l'octet des bits 7..0 et l'octet des bits 15..8. Les deux octets
comparés sont donc égaux, ce qui satisfait `octet0 == octet1` — c'est ce que vérifie
explicitement la ligne « Verification interne » de la cellule ci-dessus.

Notez la contrainte `n != 0` ajoutée explicitement : sans elle, le témoin trivial
`n = 0` aurait suffi et n'aurait rien montré. C'est le réflexe à garder devant un `sat`
qui paraît trop facile — vérifier que la requête n'admet pas une solution dégénérée
avant de lire le témoin. `MkExtract` est ce qui permet de raisonner au niveau **champ**
plutôt qu'au niveau entier : protocoles réseau, formats de fichier, registres, partout
où la question porte sur des bits précis et non sur la valeur globale.

## 7. Pourquoi la largeur fixe : BitVec vs Int

Le predicat non-trivial `a + b < a` avec `b >= 1` illustre ce que la **largeur fixe** achete :
- Sur des **bit-vectors 4 bits** (modulo 16), `a + b` peut envelopper et devenir `< a` -> le solveur trouve un temoin (`sat`).

- Sur des **entiers non bornes** (`Int`), `a + b < a` avec `b >= 1` est **impossible** -> `unsat`.



Ce contraste EST la sémantique de la théorie BitVec : sans la largeur fixe, le debordement est **inexpressible**. C'est pourquoi la verification de code machine (entiers bornes) exige BitVec, pas Int.

In [7]:
// Le meme predicat ' a + b < a, b >= 1 ' sur BitVec 4 bits vs entiers non bornes

// (a) BitVec 4 bits : l'addition enveloppe modulo 16 -> temoin possible
var ra = ctx.MkBVConst("a", 4);
var rb = ctx.MkBVConst("b", 4);
var sBv = ctx.MkSolver();
sBv.Add(ctx.MkBVUGE(rb, ctx.MkBV(1u, 4)));         // b >= 1  (unsigned)
sBv.Add(ctx.MkBVULT(ctx.MkBVAdd(ra, rb), ra));    // a + b < a  (unsigned) = debordement
var stBv = sBv.Check();
Console.WriteLine($"BV4 : ' a+b < a avec b>=1 ' -> {stBv}");
if (stBv == Status.SATISFIABLE)
{
    var mBv = sBv.Model;
    ulong wa = ((BitVecNum)mBv.Evaluate(ra)).UInt64;
    ulong wb = ((BitVecNum)mBv.Evaluate(rb)).UInt64;
    ulong sommeMod = (wa + wb) % 16;
    Console.WriteLine($"  Temoin : a={wa}, b={wb} -> (a+b) mod 16 = {sommeMod} < a={wa}  (enveloppe)");
    // Verification interne : le temoin affiche doit verifier (a+b) mod 16 < a et b >= 1
    bool temoinValide = (sommeMod < wa) && (wb >= 1);
    Console.WriteLine($"  Verification interne : (a+b) mod 16 < a ET b >= 1 -> {temoinValide}");
}
Console.WriteLine();
// (b) Entiers non bornes : aucun debordement possible
var ia = ctx.MkIntConst("ia");
var ib = ctx.MkIntConst("ib");
var sInt = ctx.MkSolver();
sInt.Add(ctx.MkGe(ia, ctx.MkInt(0)));
sInt.Add(ctx.MkGe(ib, ctx.MkInt(1)));
sInt.Add(ctx.MkLt(ctx.MkAdd(ia, ib), ia));   // a + b < a  sur les entiers
Console.WriteLine($"Int : meme predicat sur entiers non bornes -> {sInt.Check()}  (aucun debordement)");

BV4 : ' a+b < a avec b>=1 ' -> SATISFIABLE


  Temoin : a=15, b=3 -> (a+b) mod 16 = 2 < a=15  (enveloppe)


  Verification interne : (a+b) mod 16 < a ET b >= 1 -> True


Int : meme predicat sur entiers non bornes -> UNSATISFIABLE  (aucun debordement)


### Lecture : ce que la largeur fixe achète

Deux résultats opposés sur le **même** prédicat `a + b < a` avec `b >= 1` :

- en BV4, `sat`, avec un témoin vérifiant `(a + b) mod 16 < a` (le jumeau Python affichait
  `a = 13, b = 3` : `(13 + 3) mod 16 = 0`, et `0 < 13` ; le modèle affiché par le carnet
  .NET est un autre modèle parmi d'autres, vérifié par la ligne « Verification interne ») ;
- en `Int`, `unsat` : sur les entiers non bornés, ajouter un `b` positif ne peut pas
  faire diminuer `a`.

Ce couple est la démonstration la plus nette de la section. La théorie `Int` ne peut
**pas exprimer** le débordement — l'énoncé y est simplement faux — alors que `BitVec` le
rend vrai et décidable. Choisir la théorie est donc un acte de modélisation, pas un
détail d'implémentation : un débordement non modélisé n'apparaît pas comme « non
prouvé » mais comme « prouvé impossible ». C'est exactement le motif des erreurs qui
échappent à un raisonnement mené en arithmétique non bornée.

## 8. Ce que ce notebook a demontre

Quatre capacites de la théorie BitVec, inaccessibles a `Int` (non borne) ou au calcul numérique classique :
1. **Calcul exact après enveloppement** - la valeur machine reelle (3e9+1.5e9 = 205032704 après wrap), pas une approximation.

2. **Preuve d'inevitabilite** (`unsat`) - pour toute entree `x,y >= 2^31`, le debordement est force, prouve sur tout le domaine.

3. **Preuve de surete** (`unsat`) - pour toute entree `p,q < 1000`, l'addition ne deborde jamais, garantie formelle.

4. **Raisonnement au niveau bit** - extraction de champs, egalite d'octets, temoin concret produit par le solveur.



Ces quatre postures font de BitVec la théorie canonique de la **verification formelle** : prouver qu'un code (compilateur, crypto, protocole) respecte ses bornes sur toutes les entrees possibles, pas seulement sur les cas testes.

## 9. Exercices

Les trois exercices ci-dessous vous font manipuler BitVec sur des variantes : debordement signe vs non signe (1), borne exacte du debordement pour `t + t` (2), produit qui deborde (3). Chaque stub est self-contained : les primitives (`MkBVConst`, `MkBV`, `MkBVULT`/`MkBVUGE`, `MkBVAdd`/`MkBVMul`, `Solver`) sont definies dans les sections precedentes.

### Exercice 1 - Deborde signe mais pas non signe
Trouvez `a, b` tels que `a + b` deborde en **signe** (la somme devient negative, `somme < 0` signed) mais PAS en non signe (`somme >= a` unsigned reste vrai). Indice : choisissez `a, b` proches de `2^31 - 1` (la limite positive d'un int32 signe).

**Étapes** : `a = ctx.MkBV(..., 32)` ; `b = ctx.MkBV(..., 32)` ; verifier `ctx.MkBVUGE(somme, a)` vrai (pas de debord non signe) MAIS `ctx.MkBVSlt(somme, ctx.MkBV(0, 32))` ... attention, la comparaison `< 0` sur BitVec est **signée** par defaut en C# : utilisez `MkBVSlt`, pas `MkBVULT`.

In [8]:
// Exercice 1 : a+b deborde en SIGNE mais pas en non signe
// TODO etudiant :
// Etape 1 : a, b proches de 2^31 - 1 (int32 max signe)
// Etape 2 : ctx.MkBVUGE(somme, a) vrai (pas de debord non signe)
// Etape 3 : mais la somme est negative en signe (debord signe -> negatif)
//            -> ctx.MkBVSlt(somme, ctx.MkBV(0u, 32)) est vrai
(long A, long B, long Somme)? DebordementSigneVsNonSigne(Context ctx)
{
    // TODO etudiant : implementez la recherche des valeurs a, b
    return null;  // TODO etudiant : remplacer par (a, b, somme)
}

var ex1 = DebordementSigneVsNonSigne(ctx);
Console.WriteLine("Exercice 1 a completer : debordement signe vs non signe -> " + (ex1.HasValue ? ex1.Value.ToString() : "(a completer)"));


Exercice 1 a completer : debordement signe vs non signe -> (a completer)


### Exercice 2 - Borne exacte du debordement pour `t + t`
Pour `a = b = t`, a partir de quelle plus petite valeur de `t` l'addition `t + t` deborde-t-elle (en unsigned) ? Prouvez que `t >= 2^31` implique debordement, et que `t = 2^31 - 1` ne deborde pas.

**Étapes** : `t = ctx.MkBVConst("t", 32)` ; `tt = ctx.MkBVAdd(t, t)` ; `debord = ctx.MkBVULT(tt, t)` ; solver avec `ctx.MkBVUGE(t, ctx.MkBV(0x80000000u, 32))` -> `sat` ; avec `t == 2^31 - 1` -> pas de debord.

In [9]:
// Exercice 2 : borne exacte du debordement pour a=b=t
// TODO etudiant :
// Etape 1 : t = ctx.MkBVConst("t", 32) ; tt = ctx.MkBVAdd(t, t) ; debord = ctx.MkBVULT(tt, t)
// Etape 2 : solver.Add(ctx.MkBVUGE(t, ctx.MkBV(0x80000000u, 32))) et solver.Add(debord)
//           -> sat (debord a partir de 2^31)
// Etape 3 : t == 2^31 - 1 (ctx.MkBV(0x7FFFFFFFu, 32)), debord -> unsat
//           (pas de debord juste sous la borne)
long? BorneExacteDebordement(Context ctx)
{
    // TODO etudiant : implementez la recherche de la borne exacte
    return null;  // TODO etudiant : remplacer par la plus petite valeur de t qui deborde
}

var ex2 = BorneExacteDebordement(ctx);
Console.WriteLine("Exercice 2 a completer : borne exacte du debordement pour t+t -> " + (ex2.HasValue ? ex2.Value.ToString() : "(a completer)"));


Exercice 2 a completer : borne exacte du debordement pour t+t -> (a completer)


### Exercice 3 - Prouver que `65536 * 65536` deborde en BV32
`65536 = 2 puissance 16`, donc `65536 * 65536 = 4294967296` (2 puissance 32), qui en uint32 vaut exactement `0` (enveloppe complet). Prouvez que le produit deborde : `produit < 65536` (unsigned), ou directement `produit == 0`.

**Étapes** : `m1 = m2 = ctx.MkBV(65536u, 32)` ; `produit = ctx.MkBVMul(m1, m2)` ; `solver.Add(ctx.MkEq(produit, ctx.MkBV(0u, 32)))` -> `sat` (preuve que le produit enveloppe a 0).

In [10]:
// Exercice 3 : prouver que 65536 * 65536 deborde en BV32
// TODO etudiant :
// Etape 1 : m1 = ctx.MkBV(65536u, 32) ; m2 = ctx.MkBV(65536u, 32)
// Etape 2 : produit = ctx.MkBVMul(m1, m2)  (multiplication modulo 2^32)
// Etape 3 : solver.Add(ctx.MkEq(produit, ctx.MkBV(0u, 32))) -> sat
//           (preuve : 2^32 mod 2^32 = 0)
bool? PreuveProduitDeborde(Context ctx)
{
    // TODO etudiant : implementez la preuve du debordement du produit
    return null;  // TODO etudiant : remplacer par true si la preuve est etablie
}

var ex3 = PreuveProduitDeborde(ctx);
Console.WriteLine("Exercice 3 a completer : prouvez que 65536 * 65536 = 0 en BV32 -> " + (ex3.HasValue ? ex3.Value.ToString() : "(a completer)"));


Exercice 3 a completer : prouvez que 65536 * 65536 = 0 en BV32 -> (a completer)


## Conclusion
La théorie des bit-vectors de Z3 est l'outil canonique de la **verification formelle de code**. La ou `Int` (entiers non bornes) rend le debordement inexpressible, `BitVec` modelise l'arithmetique machine exacte (modulo 2^n) et permet de **prouver** des proprietes de surete ou d'inevitabilite sur tout le domaine des entrees.

Cette serie Z3-Python (notebooks 01 a 14) couvre maintenant les grandes théories du solveur : entiers et optimisation (01-11), reels exacts (12), diagnostic par UNSAT cores (13), et ici l'**arithmetique machine bornee** (14). Chacune exerce une capacite distincte qu'aucune approche numérique classique ne peut reproduire : decision exacte, non pas approximation.

## Voir aussi
- Ce carnet est le **jumeau C#** de [`Z3-14-BitVectors-Overflow-Python.ipynb`](Z3-14-BitVectors-Overflow-Python.ipynb) : meme fil pedagogique, memes verifications, API `Microsoft.Z3` (NuGet 4.12.2) au lieu de `pyz3`.
- La serie voisine [`Z3-Linq2Z3`](../Z3-Linq2Z3/README.md) rejoue le meme moteur Z3 en **C# declaratif** (Z3.Linq) : le bloc DSL `[BitVecWidth(n)]` y remplace l'appel direct a `MkBVConst`/`MkBVAdd`.

***

> Carnet issu de l'EPIC #1206.